# Benchmarking of WHI RCT and OS with selection bias

we will include all the patients who were not selected, and they will be S = 0

In [1]:
import pandas as pd 
import numpy as np 
import os 
import sys 
from tqdm import tqdm
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from scipy.stats import zscore

In [2]:
# read tables
dir_path = '/Users/zeshanmh/Documents/research/benchmarking-os/'
out   = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/outc_adj_bio.csv'))
ct_fu = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/adh_ht_pub.csv'))[['ID', 'ADHRATE', 'ENDDY', 'STARTDY', 'LOST', 'STOPHRT']] 
std_trt = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/dem_ctos_bio.csv'))[['ID', 'HRTARM', 'OSFLAG']]



In [3]:
# List of outcomes     
glbl_list = ['CHD', 'BREAST', 'STROKE', 'PE', 'ENDMTRL', 'COLORECTAL', 'BKHIP', 'DEATH']    
other_list = ['PTCA', 'DVT']

In [4]:
# Get end of follow-up for CT patients 
# BTW, do we have to consider START-DAY? what about LOST for censoring?

# keep only those with ADHRATE not missing, and group by ID to get max ENDDY
# keep columns 'ID', 'ENDDY', and 'LOST'
# rename ENDDY to END_DY
# ct_end = ct_fu[ct_fu['ADHRATE'].notna()].groupby('ID')['ENDDY'].max().reset_index() 
# ct_end = ct_end.rename(columns={'ENDDY': 'END_DY'})
# ct_end

# ct_end = ct_fu[ct_fu['ADHRATE'].notna()][['ADHRATE','ID', 'ENDDY', 'LOST']].rename(columns={'ENDDY': 'END_DY'})
# ct_end = ct_fu[['ADHRATE','ID', 'ENDDY', 'LOST']].rename(columns={'ENDDY': 'END_DY'})
# ct_end = ct_end.query('ADHRATE != 0.')[['ID','END_DY','LOST']]

## OG
# ct_end = ct_fu[ct_fu['ADHRATE'].notna()].groupby('ID')['ENDDY'].max().reset_index() 
# ct_end = ct_end.rename(columns={'ENDDY': 'END_DY'})
# ct_end
def get_lost_day(group):
    lost_rows = group[group['LOST'] == 'Yes']
    return lost_rows['ENDDY'].iloc[0] if not lost_rows.empty else None

ct_end = (ct_fu[ct_fu['ADHRATE'].notna()]
          .groupby('ID')
          .agg({
              'ENDDY': 'max',
              'LOST': lambda x: 1 if 'Yes' in x.values else 0,
          })
          .reset_index())

# ADD LOST_DY column
lost_days = (ct_fu[ct_fu['ADHRATE'].notna()]
             .groupby('ID')[['LOST','ENDDY']]
             .apply(get_lost_day)
             .rename('LOST_DY'))

ct_end = ct_end.merge(lost_days.to_frame(), on='ID', how='left')
ct_end = ct_end.rename(columns={'ENDDY': 'END_DY'})
ct_end


,ID,END_DY,LOST,LOST_DY
0,500001,2190,0,NaN
1,500022,3287,0,NaN
2,500024,1095,0,NaN
3,500025,2190,0,NaN
4,500027,2921,0,NaN
...,...,...,...,...
27168,699951,2556,0,NaN
27169,699963,2556,0,NaN
27170,699974,1460,0,NaN
27171,699987,4017,0,NaN


In [5]:
ct_end[ct_end['LOST'] == 1].shape

(205, 4)

In [6]:
ct_df = std_trt.drop_duplicates('ID')
ct_df = ct_df[ct_df['HRTARM'].isin(['E+P intervention', 'E+P control'])]
ct_df = ct_df.merge(ct_end, on='ID', how='left')
ct_df = ct_df.merge(out, on='ID', how='left')

# code variables HRTARM and OS 
ct_df['OS'] = 0 
ct_df['HRTARM'] = ct_df['HRTARM'].map({'E+P intervention': 1, 'E+P control': 0})

# print out first 10 rows
print(ct_df.shape)
print(ct_df[ct_df['HRTARM'] == 1].shape)
print(ct_df[ct_df['HRTARM'] == 0].shape)
ct_df.head(n=10)


(16608, 360)
(8506, 360)
(8102, 360)


,ID,HRTARM,OSFLAG,END_DY,LOST,LOST_DY,ANGINA,ANGINADY,ANGINASRC,AANEUR,...,DEATHDY,DEATHSRC,DEATHCAUSESRC,HYST,HYSTDY,HYSTSRC,ENDWHIDY,ENDEXT1DY,ENDFOLLOWDY,OS
0,642629,1,No,1460.0,0.0,NaN,0,NaN,NaN,0,...,NaN,NaN,NaN,1,4499.0,1.0,3480.0,5481.0,9083.0,0
1,568085,1,No,1825.0,0.0,NaN,0,NaN,NaN,0,...,7610.0,2.0,1.0,0,NaN,NaN,2725.0,4726.0,7610.0,0
2,568186,0,No,2555.0,0.0,NaN,0,NaN,NaN,0,...,NaN,NaN,NaN,0,NaN,NaN,3138.0,5139.0,8013.0,0
3,623255,1,No,1825.0,0.0,NaN,0,NaN,NaN,0,...,NaN,NaN,NaN,0,NaN,NaN,2500.0,4501.0,8015.0,0
4,537848,1,No,2555.0,0.0,NaN,0,NaN,NaN,0,...,NaN,NaN,NaN,0,NaN,NaN,3263.0,5264.0,8992.0,0
5,668539,1,No,2556.0,0.0,NaN,0,NaN,NaN,0,...,NaN,NaN,NaN,0,NaN,NaN,3486.0,5487.0,5487.0,0
6,660370,1,No,1095.0,0.0,NaN,0,NaN,NaN,0,...,827.0,1.0,1.0,0,NaN,NaN,827.0,827.0,827.0,0
7,551999,1,No,1094.0,0.0,NaN,0,NaN,NaN,0,...,3926.0,1.0,1.0,0,NaN,NaN,3012.0,3926.0,3926.0,0
8,682433,0,No,1095.0,0.0,NaN,0,NaN,NaN,0,...,NaN,NaN,NaN,1,834.0,0.0,2521.0,4522.0,8021.0,0
9,605069,1,No,1460.0,0.0,NaN,0,NaN,NaN,0,...,NaN,NaN,NaN,0,NaN,NaN,2681.0,4682.0,8375.0,0


In [7]:
diff_selection_for_CT = False

# process outcomes 
for i in glbl_list + other_list: 
    ct_df[i+'_E']  = ((ct_df[i] == 1) & (ct_df[i+'DY'] <= ct_df['END_DY'])).astype(int)
    ct_df[i+'_DY'] = np.where(ct_df[i+'_E'] == 1, ct_df[i+'DY'], ct_df['END_DY'])
    ct_df[i+'_EDY'] = np.where(ct_df[i+'_E'] == 1, ct_df[i+'DY'], np.nan) 

# Global index
ct_df['GLBL_E'] = (ct_df[[j+'_E' for j in glbl_list]].sum(axis=1) > 0).astype(int)
ct_df['GLBL_DY'] = np.where(ct_df['GLBL_E'] == 1,
                            ct_df[[j+'_EDY' for j in glbl_list]].min(axis=1),
                            ct_df[[j+'_DY' for j in glbl_list]].min(axis=1))

# Selection variable 
ct_df['S'] = 1

# Add different selection variables for each outcome (this is because S = 0 for censored patients)
for i in glbl_list + other_list: 
    ct_df['S_'+i] = ct_df['S']
    if diff_selection_for_CT:
        ct_df['S_'+i] = np.where(ct_df[i+'DY'] > ct_df['END_DY'], 0, ct_df['S_'+i])
        ct_df['S_'+i] = np.where(((ct_df['LOST'] == 1) & (ct_df[i+'DY'] > ct_df['LOST_DY'])), 0, ct_df['S_'+i])
ct_df['S_GLBL'] = ct_df['S']

# Select needed columns
ct_df = ct_df[['ID', 'OS', 'HRTARM'] + 
                ['S_'+j for j in glbl_list + other_list + ['GLBL']] +
                [j+'_E' for j in glbl_list + other_list + ['GLBL']] + 
                [j+'_DY' for j in glbl_list + other_list + ['GLBL']]]


In [8]:
ct_df[ct_df['S_STROKE'] == 0].shape

(0, 36)

In [9]:
ct_df.query('HRTARM == 0 & STROKE_E == 1')

,ID,OS,HRTARM,S_CHD,S_BREAST,S_STROKE,S_PE,S_ENDMTRL,S_COLORECTAL,S_BKHIP,...,BREAST_DY,STROKE_DY,PE_DY,ENDMTRL_DY,COLORECTAL_DY,BKHIP_DY,DEATH_DY,PTCA_DY,DVT_DY,GLBL_DY
340,686812,0,0,1,1,1,1,1,1,1,...,1460.0,1709.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,1460.0
502,689189,0,0,1,1,1,1,1,1,1,...,1825.0,1577.0,1825.0,1825.0,1825.0,1825.0,1577.0,1825.0,1825.0,1577.0
702,500221,0,0,1,1,1,1,1,1,1,...,1825.0,190.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,190.0
888,669029,0,0,1,1,1,1,1,1,1,...,1825.0,1797.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,1797.0
1337,552393,0,0,1,1,1,1,1,1,1,...,1095.0,597.0,1095.0,1095.0,1095.0,1095.0,1095.0,1095.0,1095.0,597.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15576,681806,0,0,1,1,1,1,1,1,1,...,1825.0,1550.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,1825.0,1550.0
15664,515015,0,0,1,1,1,1,1,1,1,...,2921.0,2136.0,2921.0,2921.0,2921.0,2921.0,2921.0,2921.0,2921.0,2136.0
16008,643485,0,0,1,1,1,1,1,1,1,...,2555.0,1292.0,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0,2555.0,1292.0
16406,598915,0,0,1,1,1,1,1,1,1,...,1095.0,738.0,1095.0,1095.0,1095.0,1095.0,1095.0,1095.0,1095.0,738.0


## OS Specification (SELECTION FLAG + CENSORED)

In [12]:
dir_path = '/Users/zeshanmh/Documents/research/benchmarking-os/'
hyst    = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/f2_ctos_bio.csv'))[['ID','HYST']]
pre_hrt  = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/f43_ctos_bio.csv'))[['ID', 'TOTESTAT','TOTPSTAT']]
post_hrt = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/f48_av1_os_pub.csv'))[['ID','ELSTYR','PLSTYR','HRTCMBP']]
unc_hf   = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/unc_hf_bio.csv'))[['ID','CHDYRHX','CHDEVERHX','HYPERTNHX','MIHX','PVDHX','DIABHX','STROKEHX']]

/var/folders/t9/9775q6dn21l67f71h7t7xj0h0000gn/T/ipykernel_25007/1319091169.py:2: DtypeWarning: Columns (39,42,57,59,66) have mixed types. Specify dtype option on import or set low_memory=False.
  hyst    = pd.read_csv(os.path.join(dir_path, 'whi/data/data/main_study/csv/f2_ctos_bio.csv'))[['ID','HYST']]


In [13]:
# construct os_df 
selection_flag = 'biased'
'''
drop_all_excluded: this drops all patients who had hysterectomy OR are on unopposed estrogen; thus, selection, S = 0 and S = 1, is based on censoring only
drop_some_excluded: this keeps patients who had hyseterectomy OR are on unopposed estrogen but were past users of combined HRT, assigns them to be S = 0;
censored patients are additionally S = 0
drop_no_excluded: keeps all patients who had hysterectomy OR are on unopposed estrogen, and assigns them S = 0; censored patients are additionally S = 0
'''
additional_selection_processing = 'drop_all_excluded' # 'drop_some_excluded', 'drop_no_excluded', 'drop_all_excluded'
'''
if censored_patients_sel0 = True, then censored patients are additionally S = 0
'''
censored_patients_sel0 = True 

os_df = std_trt.drop_duplicates('ID')
os_df = os_df[os_df['OSFLAG'] == 'Yes']
os_df = os_df.merge(hyst, on='ID', how='left')
os_df = os_df.merge(pre_hrt, on='ID', how='left')
print(os_df['TOTESTAT'].value_counts())
print(os_df['HYST'].value_counts())
os_df = os_df.merge(post_hrt, on='ID', how='left')
if additional_selection_processing == 'drop_some_excluded': 
    os_df = os_df.merge(unc_hf, on='ID', how='left')
    condition_dict = {
        'CHDEVERHX': ('!=', 1.),
        'HYPERTNHX': ('!=', 1.),
        'MIHX': ('!=', 1.),
        'PVDHX': ('!=', 1.),
        'DIABHX': ('!=', 1.),
        'STROKEHX': ('!=', 1.)
    }
    
    # condition = (os_df['HYST'] == 'Yes') & (
    #     pd.concat([
    #         os_df[var].apply(lambda x: eval(f"x {op} {repr(val)}"))
    #         for var, (op, val) in condition_dict.items()
    #     ], axis=1).all(axis=1) | (os_df['TOTPSTAT'] == 'Never used')
    # ) 
    # os_df = os_df[~condition]
    # condition2 = (os_df['TOTESTAT'] == 'Current user') & (
    #     pd.concat([
    #         os_df[var].apply(lambda x: eval(f"x {op} {repr(val)}"))
    #         for var, (op, val) in condition_dict.items()
    #     ], axis=1).all(axis=1) | (os_df['TOTPSTAT'] == 'Never used')
    # )
    # os_df = os_df[~condition2]
    condition = (os_df['HYST'] == 'Yes') & ((os_df['TOTPSTAT'] == 'Never used') | (os_df['TOTPSTAT'] == 'Current user'))
    os_df = os_df[~condition]
    condition2 = (os_df['TOTESTAT'] == 'Current user') & ((os_df['TOTPSTAT'] == 'Never used') | (os_df['TOTPSTAT'] == 'Current user'))
    os_df = os_df[~condition2]
    os_df['S'] = os_df.apply(
        lambda row: 0 if (row['HYST'] == 'Yes' or row['TOTESTAT'] == 'Current user') else 1,
        axis=1
    )
elif additional_selection_processing == 'drop_no_excluded': 
    # Selected patients
    os_df['S'] = os_df.apply(
        lambda row: 1 if (row['HYST'] == 'No' and row['TOTESTAT'] in ['Never used', 'Past user']) else 0,
        axis=1
    )
elif additional_selection_processing == 'drop_all_excluded':
    os_df = os_df[os_df['HYST'] == 'No']
    os_df = os_df[os_df['TOTESTAT'].isin(['Never used', 'Past user'])]
    os_df['S'] = 1

os_df = os_df.merge(out, on='ID', how='left')

# 35551 (control) + 17503 (intervention) = 53054

print(os_df[os_df['TOTPSTAT'].isin(['Current user'])].shape)
print(os_df[os_df['TOTPSTAT'].isin(['Never used', 'Past user'])].shape)

if selection_flag == 'biased': 
    os_df = os_df[os_df['TOTPSTAT'].isin(['Never used', 'Past user','Current user'])]
    os_df['HRTARM'] = os_df['TOTPSTAT'].map({'Current user': 1, 'Never used': 0, 'Past user': 0})
elif selection_flag == 'unbiased' or selection_flag == 'manually_biased': 
    os_df = os_df[os_df['TOTPSTAT'].isin(['Never used', 'Past user','Current user'])]
    conditions = [
        (((os_df['ELSTYR'] == 'Yes') & (os_df['PLSTYR'] == 'Yes')) | (os_df['HRTCMBP'] == 'Yes')),
        ((os_df['ELSTYR'] == 'No') & (os_df['PLSTYR'] == 'No')),
        (((os_df['ELSTYR'] == 'Yes') & (os_df['PLSTYR'] == 'No')) | ((os_df['ELSTYR'] == 'No') & (os_df['PLSTYR'] == 'Yes')))
    ]
    choices = [1, 0, -1]
    os_df['HRTGRP'] = np.select(conditions, choices, default=-2)
    os_df = os_df[os_df['HRTGRP'] != -2]
    os_df['HRTARM'] = (os_df['HRTGRP'] == 1).astype(int)
    os_df['S'] = os_df.apply(lambda row: 0 if row['TOTPSTAT'] == 'Current user' else row['S'], axis=1)
os_df['OS'] = 1

# os_end_day = None
os_end_day = 6*365
os_df['END_DY'] = os_end_day if os_end_day is not None else os_df['ENDFOLLOWDY']
# os_df['END_DY'] = os_df.apply(lambda x: x['DEATHDY'] if x['DEATHDY'] < os_end_day else os_end_day, axis=1)


# Process outcomes (same as CT)
for i in glbl_list + other_list:
    os_df[i+'_E'] = ((os_df[i] == 1) & (os_df[i+'DY'] <= os_df['END_DY'])).astype(int)
    os_df[i+'_DY'] = np.where(os_df[i+'_E'] == 1, os_df[i+'DY'], os_df['END_DY'])
    os_df[i+'_EDY'] = np.where(os_df[i+'_E'] == 1, os_df[i+'_DY'], np.nan)

# Global index
os_df['GLBL_E'] = (os_df[[j+'_E' for j in glbl_list]].sum(axis=1) > 0).astype(int)
os_df['GLBL_DY'] = np.where(os_df['GLBL_E'] == 1,
                            os_df[[j+'_EDY' for j in glbl_list]].min(axis=1),
                            os_df[[j+'_DY' for j in glbl_list]].min(axis=1))

# Selection variable adjustment
for i in glbl_list + other_list:
    os_df['S_'+i] = os_df['S']
    if censored_patients_sel0: 
        os_df['S_'+i] = np.where(os_df[i+'DY'] > os_df['END_DY'], 0, os_df['S_'+i])
os_df['S_GLBL'] = os_df['S']

# Select needed columns
os_df = os_df[['ID', 'OS', 'HRTARM'] + 
                ['S_'+j for j in glbl_list + other_list + ['GLBL']] + 
                [j+'_E' for j in glbl_list + other_list + ['GLBL']] + 
                [j+'_DY' for j in glbl_list + other_list + ['GLBL']]]

os_df

TOTESTAT
Never used      58890
Current user    23290
Past user       11432
Name: count, dtype: int64
HYST
No     54440
Yes    39149
Name: count, dtype: int64
(17509, 363)
(35539, 363)


,ID,OS,HRTARM,S_CHD,S_BREAST,S_STROKE,S_PE,S_ENDMTRL,S_COLORECTAL,S_BKHIP,...,BREAST_DY,STROKE_DY,PE_DY,ENDMTRL_DY,COLORECTAL_DY,BKHIP_DY,DEATH_DY,PTCA_DY,DVT_DY,GLBL_DY
0,591800,1,0,1,1,1,1,0,1,1,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
1,548198,1,0,1,1,1,1,1,1,1,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
2,670866,1,1,1,1,1,1,1,1,1,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
3,665473,1,1,1,1,1,1,1,1,1,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
4,508135,1,0,1,1,1,1,1,1,1,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53071,641028,1,0,1,1,1,1,0,1,1,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
53072,687963,1,0,1,1,1,1,1,1,1,...,203.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,203.0
53073,681994,1,1,1,1,1,1,1,1,1,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0
53074,575472,1,0,1,1,1,1,1,1,1,...,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0,2190.0


In [14]:
print(os_df[os_df['S_CHD'] == 0].shape)
print(os_df[os_df['S_CHD'] == 1].shape)

(1457, 36)
(51591, 36)


# Setup of dataframes with target variables

In [16]:
ctos_df = pd.concat([ct_df, os_df], ignore_index=True)

In [19]:
if selection_flag == 'manually_biased': 
    # removing age and menopausal status
    categorical_features = {
        'dem_ctos_bio.csv': {'ETHNIC': True, 'EDUC': True}, 
        'f80_ctos_bio.csv': {'BMI': False}, 
        'f34_ctos_bio.csv': {'SMOKING': True}, 
        'f151_ctos_bio.csv': {'PHYSFUN': False}    
    }
    
    new_feature_dict = { 
        'dem_ctos_bio.csv': ['ETHNIC_White', \
                             'EDUC_Some post-graduate or professional', \
                             'EDUC_Some college or Associate Degree'],
        'f80_ctos_bio.csv': ['BMI'],
        'f34_ctos_bio.csv': ['SMOKING_Past Smoker', 'SMOKING_Current Smoker'],
        'f151_ctos_bio.csv': ['PHYSFUN']
    }
else:
    categorical_features = {
        'dem_ctos_bio.csv': {'AGE': False, 'ETHNIC': True, 'EDUC': True}, 
        'f80_ctos_bio.csv': {'BMI': False}, 
        'f34_ctos_bio.csv': {'SMOKING': True}, 
        'f31_ctos_bio.csv': {'MENO': False}, 
        'f151_ctos_bio.csv': {'PHYSFUN': False}    
    }
    
    new_feature_dict = { 
        'dem_ctos_bio.csv': ['AGE', 'ETHNIC_White', \
                             'EDUC_Some post-graduate or professional', \
                             'EDUC_Some college or Associate Degree'],
        'f80_ctos_bio.csv': ['BMI'],
        'f34_ctos_bio.csv': ['SMOKING_Past Smoker', 'SMOKING_Current Smoker'],
        'f31_ctos_bio.csv': ['MENO'],
        'f151_ctos_bio.csv': ['PHYSFUN']
    }
    

In [20]:
import pandas.api.types as ptypes

ctos_temp = ctos_df.copy()
# Dictionary to specify which features are categorical

# dfs = []  # Store all dataframes to concatenate later
new_dir_path = dir_path + 'whi/data/data/main_study/csv'

for filename, f_dict in categorical_features.items():
    # Read the data
    df = pd.read_csv(os.path.join(new_dir_path, filename))
    if filename == 'f80_ctos_bio.csv': 
        df = df.query('F80VTYP == "Screening"')
    elif filename == 'f151_ctos_bio.csv': 
        idx = df.groupby('ID')['F151DAYS'].idxmin().reset_index(drop=True)
        df = df.loc[idx, :].reset_index(drop=True)[['ID','PHYSFUN']]
    # Select needed columns
    features = list(f_dict.keys())
    df = df[['ID'] + features]
    
    # Separate ID column
    id_col = df['ID']
    print(f"Processed {filename}")
    print(df.shape)

    orig_cols = ctos_temp.columns.tolist()
    ctos_temp = ctos_temp.merge(df, on='ID', how='left')

    # Handle continuous and categorical features separately
    cont_features = [f for f in features if not f_dict[f]]
    cat_features = [f for f in features if f_dict[f]]
    
    # Handle continuous features
    if cont_features:
        cont_imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
        ctos_temp[cont_features] = cont_imputer.fit_transform(ctos_temp[cont_features])
    
    # Handle categorical features
    if cat_features:
        cat_imputer = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
        ctos_temp[cat_features] = cat_imputer.fit_transform(ctos_temp[cat_features])
        
        # One-hot encode categorical features
        ctos_temp = pd.get_dummies(ctos_temp, columns=cat_features, prefix=cat_features)

    if filename == 'dem_ctos_bio.csv': 
        ctos_temp = ctos_temp.rename(columns={'ETHNIC_White (not of Hispanic origin)': 'ETHNIC_White'})

    ctos_temp = ctos_temp[orig_cols + new_feature_dict[filename]]

ctos_temp = ctos_temp.astype({col: int for col in ctos_temp.select_dtypes(include='bool').columns})
display(ctos_temp)    


Processed dem_ctos_bio.csv
(161808, 4)
Processed f80_ctos_bio.csv
(161771, 2)
Processed f34_ctos_bio.csv
(161625, 2)


/var/folders/t9/9775q6dn21l67f71h7t7xj0h0000gn/T/ipykernel_25007/1128475091.py:11: DtypeWarning: Columns (20,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(new_dir_path, filename))


Processed f31_ctos_bio.csv
(161705, 2)
Processed f151_ctos_bio.csv
(113491, 2)


,ID,OS,HRTARM,S_CHD,S_BREAST,S_STROKE,S_PE,S_ENDMTRL,S_COLORECTAL,S_BKHIP,...,GLBL_DY,AGE,ETHNIC_White,EDUC_Some post-graduate or professional,EDUC_Some college or Associate Degree,BMI,SMOKING_Past Smoker,SMOKING_Current Smoker,MENO,PHYSFUN
0,642629,0,1,1,1,1,1,1,1,1,...,935.0,64.0,1,0,0,29.19411,0,0,54.0,65.000000
1,568085,0,1,1,1,1,1,1,1,1,...,1825.0,62.0,0,0,0,19.55943,0,1,51.0,90.000000
2,568186,0,0,1,1,1,1,1,1,1,...,2555.0,62.0,1,0,1,30.44928,1,0,44.0,50.000000
3,623255,0,1,1,1,1,1,1,1,1,...,1825.0,60.0,1,1,0,28.54828,0,0,54.0,78.137062
4,537848,0,1,1,1,1,1,1,1,1,...,2555.0,54.0,1,0,1,40.32766,1,0,54.0,65.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69651,641028,1,0,1,1,1,1,0,1,1,...,2190.0,69.0,1,1,0,25.50301,0,0,54.0,90.000000
69652,687963,1,0,1,1,1,1,1,1,1,...,203.0,70.0,1,0,1,23.11101,1,0,55.0,60.000000
69653,681994,1,1,1,1,1,1,1,1,1,...,2190.0,53.0,1,0,0,26.76123,0,0,48.0,95.000000
69654,575472,1,0,1,1,1,1,1,1,1,...,2190.0,75.0,1,0,0,28.07091,1,0,50.0,78.137062


## Quick Analysis

In [29]:
# hazard ratios for stroke, breast cancer, and CHD in clinical trial vs observational study 

## CT 
ct_df = ctos_temp.query('OS == 0 & S_GLBL == 1')
ct_df_sub = ct_df[['ID','HRTARM', 'STROKE_E', 'BREAST_E', 'CHD_E','STROKE_DY', 'BREAST_DY', 'CHD_DY']]
ct_df_chd = ct_df[['HRTARM', 'CHD_E', 'CHD_DY']]
ct_df_chd = ct_df_chd[ct_df_chd['CHD_DY'].notna()]

ct_df_stroke = ct_df[['HRTARM', 'STROKE_E', 'STROKE_DY']]
ct_df_stroke = ct_df_stroke[ct_df_stroke['STROKE_DY'].notna()]

ct_df_breast = ct_df[['HRTARM', 'BREAST_E', 'BREAST_DY']]
ct_df_breast = ct_df_breast[ct_df_breast['BREAST_DY'].notna()]

from lifelines import CoxPHFitter

def get_hr(df, Y, E, event_name, HR_cov='HRTARM', study_type='Clinical Trial'): 
    cph = CoxPHFitter()
    cph.fit(df, duration_col=Y, event_col=E)
    cph.print_summary()
    cHR = cph.hazard_ratios_[HR_cov]
    cis = cph.confidence_intervals_
    lower = np.exp(cis['95% lower-bound'][HR_cov])
    upper = np.exp(cis['95% upper-bound'][HR_cov])
    print(f'Hazard ratio for {event_name} in {study_type}: {np.round(cHR, 2)} (95% CI: {np.round(lower, 2)}, {np.round(upper, 2)})')

get_hr(ct_df_chd, 'CHD_DY', 'CHD_E', 'CHD')
get_hr(ct_df_stroke, 'STROKE_DY', 'STROKE_E', 'Stroke')
get_hr(ct_df_breast, 'BREAST_DY', 'BREAST_E', 'Breast Cancer')


<lifelines.CoxPHFitter: fitted with 16516 total observations, 16218 right-censored observations>
             duration col = 'CHD_DY'
                event col = 'CHD_E'
      baseline estimation = breslow
   number of observations = 16516
number of events observed = 298
   partial log-likelihood = -2787.04
         time fit was run = 2025-02-24 21:53:43 UTC

---
           coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                  
HRTARM     0.24      1.28      0.12            0.01            0.47                1.01                1.61

           cmp to    z    p  -log2(p)
covariate                            
HRTARM       0.00 2.09 0.04      4.76
---
Concordance = 0.54
Partial AIC = 5576.08
log-likelihood ratio test = 4.39 on 1 df
-log2(p) of ll-ratio test = 4.79

Hazard ratio for CHD in Clinical Trial: 1.28 (95% CI: 1.01, 1.61)


<lifelines.CoxPHFitter: fitted with 16516 total observations, 16304 right-censored observations>
             duration col = 'STROKE_DY'
                event col = 'STROKE_E'
      baseline estimation = breslow
   number of observations = 16516
number of events observed = 212
   partial log-likelihood = -1970.95
         time fit was run = 2025-02-24 21:53:43 UTC

---
           coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                  
HRTARM     0.32      1.37      0.14            0.04            0.59                1.04                1.80

           cmp to    z    p  -log2(p)
covariate                            
HRTARM       0.00 2.27 0.02      5.41
---
Concordance = 0.53
Partial AIC = 3943.90
log-likelihood ratio test = 5.21 on 1 df
-log2(p) of ll-ratio test = 5.47

Hazard ratio for Stroke in Clinical Trial: 1.37 (95% CI: 1.04, 1.8)


<lifelines.CoxPHFitter: fitted with 16516 total observations, 16116 right-censored observations>
             duration col = 'BREAST_DY'
                event col = 'BREAST_E'
      baseline estimation = breslow
   number of observations = 16516
number of events observed = 400
   partial log-likelihood = -3713.55
         time fit was run = 2025-02-24 21:53:43 UTC

---
           coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                  
HRTARM     0.29      1.34      0.10            0.09            0.49                1.09                1.63

           cmp to    z      p  -log2(p)
covariate                              
HRTARM       0.00 2.86 <0.005      7.86
---
Concordance = 0.52
Partial AIC = 7429.11
log-likelihood ratio test = 8.25 on 1 df
-log2(p) of ll-ratio test = 7.94

Hazard ratio for Breast Cancer in Clinical Trial: 1.34 (95% CI: 1.09, 1.63)


In [34]:
# OS 
os_df = ctos_temp.query('OS == 1 & S_CHD == 1')
features = ['AGE','ETHNIC_White', 'EDUC_Some post-graduate or professional', \
            'EDUC_Some college or Associate Degree', 'BMI', 'SMOKING_Past Smoker', \
            'SMOKING_Current Smoker', 'MENO', 'PHYSFUN']
treatment = ['HRTARM']
events = ['CHD_E', 'CHD_DY']
event_name = 'CHD'
# events = ['STROKE_E', 'STROKE_DY']
# event_name = 'STROKE'

os_df_sub = os_df[features + treatment + events]
os_df_sub = os_df_sub[os_df_sub[events[1]].notna()]

get_hr(os_df_sub, events[1], events[0], event_name, HR_cov='HRTARM', study_type='Observational Study')




<lifelines.CoxPHFitter: fitted with 51591 total observations, 50704 right-censored observations>
             duration col = 'CHD_DY'
                event col = 'CHD_E'
      baseline estimation = breslow
   number of observations = 51591
number of events observed = 887
   partial log-likelihood = -9322.20
         time fit was run = 2025-02-24 21:53:44 UTC

---
                                         coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                                
AGE                                      0.10      1.11      0.01            0.09            0.11                1.09                1.12
ETHNIC_White                             0.01      1.01      0.10           -0.18            0.21                0.84                1.23
EDUC_Some post-graduate or professional -0.25      0.78      0.11           -0.47           -0.02                0.62                0.98
EDUC_Some college or Associate Degree   -0.21      0.81      0.08           -0.37           -0.05                0.69                0.95
BMI                                      0.04      1.05      0.00            0.03            0.05                1.04                1.06
SMOKING_Past Smoker                      0.28      1.32      0.07            0.14            0.41                1.15                1.51
SMOKING_Current Smoker                   0.66      1.94      0.13            0.41            0.92                1.50                2.50
MENO                                    -0.02      0.98      0.01           -0.03           -0.01                0.97                0.99
PHYSFUN                                 -0.00      1.00      0.00           -0.01           -0.00                0.99                1.00
HRTARM                                  -0.14      0.87      0.09           -0.31            0.03                0.73                1.03

                                         cmp to     z      p  -log2(p)
covariate                                                             
AGE                                        0.00 18.82 <0.005    260.08
ETHNIC_White                               0.00  0.15   0.88      0.18
EDUC_Some post-graduate or professional    0.00 -2.17   0.03      5.07
EDUC_Some college or Associate Degree      0.00 -2.56   0.01      6.59
BMI                                        0.00  8.97 <0.005     61.59
SMOKING_Past Smoker                        0.00  3.91 <0.005     13.38
SMOKING_Current Smoker                     0.00  5.10 <0.005     21.46
MENO                                       0.00 -2.81 <0.005      7.66
PHYSFUN                                    0.00 -2.82 <0.005      7.69
HRTARM                                     0.00 -1.66   0.10      3.38
---
Concordance = 0.73
Partial AIC = 18664.41
log-likelihood ratio test = 590.13 on 10 df
-log2(p) of ll-ratio test = 397.43

Hazard ratio for CHD in Observational Study: 0.87 (95% CI: 0.73, 1.03)


In [35]:
# OS 
os_df = ctos_temp.query('OS == 1 & S_STROKE == 1')
features = ['AGE','ETHNIC_White', 'EDUC_Some post-graduate or professional', \
            'EDUC_Some college or Associate Degree', 'BMI', 'SMOKING_Past Smoker', \
            'SMOKING_Current Smoker', 'MENO', 'PHYSFUN']
treatment = ['HRTARM']
events = ['STROKE_E', 'STROKE_DY']
event_name = 'STROKE'

os_df_sub = os_df[features + treatment + events]
os_df_sub = os_df_sub[os_df_sub[events[1]].notna()]

get_hr(os_df_sub, events[1], events[0], event_name, HR_cov='HRTARM', study_type='Observational Study')




<lifelines.CoxPHFitter: fitted with 51891 total observations, 51207 right-censored observations>
             duration col = 'STROKE_DY'
                event col = 'STROKE_E'
      baseline estimation = breslow
   number of observations = 51891
number of events observed = 684
   partial log-likelihood = -7192.59
         time fit was run = 2025-02-24 21:53:44 UTC

---
                                         coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                                
AGE                                      0.11      1.11      0.01            0.09            0.12                1.10                1.12
ETHNIC_White                            -0.12      0.89      0.11           -0.33            0.10                0.72                1.10
EDUC_Some post-graduate or professional -0.03      0.97      0.12           -0.27            0.21                0.76                1.23
EDUC_Some college or Associate Degree   -0.13      0.88      0.09           -0.31            0.05                0.73                1.05
BMI                                      0.02      1.02      0.01            0.01            0.03                1.01                1.03
SMOKING_Past Smoker                      0.08      1.08      0.08           -0.08            0.24                0.92                1.27
SMOKING_Current Smoker                   0.73      2.08      0.14            0.46            1.00                1.59                2.72
MENO                                    -0.00      1.00      0.01           -0.02            0.01                0.98                1.01
PHYSFUN                                 -0.01      0.99      0.00           -0.01           -0.00                0.99                1.00
HRTARM                                  -0.15      0.86      0.10           -0.34            0.04                0.71                1.04

                                         cmp to     z      p  -log2(p)
covariate                                                             
AGE                                        0.00 17.10 <0.005    215.35
ETHNIC_White                               0.00 -1.08   0.28      1.84
EDUC_Some post-graduate or professional    0.00 -0.26   0.79      0.33
EDUC_Some college or Associate Degree      0.00 -1.43   0.15      2.71
BMI                                        0.00  2.97 <0.005      8.38
SMOKING_Past Smoker                        0.00  1.00   0.32      1.65
SMOKING_Current Smoker                     0.00  5.34 <0.005     23.33
MENO                                       0.00 -0.30   0.77      0.38
PHYSFUN                                    0.00 -4.07 <0.005     14.37
HRTARM                                     0.00 -1.53   0.13      2.98
---
Concordance = 0.73
Partial AIC = 14405.18
log-likelihood ratio test = 458.01 on 10 df
-log2(p) of ll-ratio test = 303.59

Hazard ratio for STROKE in Observational Study: 0.86 (95% CI: 0.71, 1.04)


## Adding target variables (CHD/STROKE + LR/RF)

In [ ]:
# A, Y, R, S, X
ctos_temp.columns

In [57]:
from sklearn.model_selection import train_test_split 

df_ctos = ctos_temp.copy()
if selection_flag == 'manually_biased': 
    predictors = ['ETHNIC_White', 'EDUC_Some post-graduate or professional', 
          'EDUC_Some college or Associate Degree', 'BMI', 'SMOKING_Past Smoker', 
          'SMOKING_Current Smoker', 'PHYSFUN'] 
else: 
    predictors = ['AGE', 'ETHNIC_White', 'EDUC_Some post-graduate or professional', 
          'EDUC_Some college or Associate Degree', 'BMI', 'SMOKING_Past Smoker', 
          'SMOKING_Current Smoker', 'MENO', 'PHYSFUN'] 
outcome_name = 'CHD' # STROKE, BREAST

outcome= outcome_name + '_E'
trt    = 'HRTARM'
select = f'S_{outcome_name}'

drop_columns = [x for x in df_ctos.columns if x not in predictors + [outcome, trt, 'ID', 'S']]
df_ctos.rename(columns={trt: 'A', outcome: 'Y'}, inplace=True) 
df_ctos['S']  = df_ctos[select]
df_ctos['Y0'] = df_ctos['Y']
df_ctos['Y1'] = df_ctos['Y']
df_ctos['R'] = 1 - df_ctos['OS']
df_ctos.drop(columns=drop_columns, inplace=True)

seeds = [42]
seeds += [x for x in range(19)]

df_rct_train_list = []
df_obs_train_list = []
df_rct_val_list = [] 
df_obs_val_list = []

for seed in seeds:  
    # split into train and val
    df_ctos_train, df_ctos_val = train_test_split(df_ctos, test_size=0.25, random_state=seed)

    # split into RCT and OBS
    df_rct_train = df_ctos_train.query('R == 1') 
    df_obs_train = df_ctos_train.query('R == 0')
    df_rct_val   = df_ctos_val.query('R == 1')
    df_obs_val   = df_ctos_val.query('R == 0')

    # add into lists
    df_rct_train_list.append(df_rct_train)
    df_obs_train_list.append(df_obs_train)
    df_rct_val_list.append(df_rct_val) 
    df_obs_val_list.append(df_obs_val)
    

In [ ]:
df_rct_train_list[9]

## Training models (multiple trials, $n=20$)

In [61]:
import sys
sys.path.append('../synthetic/')
from utils_data_v2 import fit_models, make_preds, merge_df_val, fit_model
from collections import defaultdict
from utils_v2 import pearsonr

In [ ]:
num_trials = len(seeds)
bias_res = list() 
cov_res = defaultdict(lambda: defaultdict(list))
from tqdm import tqdm
model_type = 'RF'
for i in tqdm(range(num_trials)): 
    df_rct_train = df_rct_train_list[i]
    df_obs_train = df_obs_train_list[i]
    df_rct_val   = df_rct_val_list[i]
    df_obs_val   = df_obs_val_list[i]

    rct_models = fit_models(df_rct_train, predictors, is_rct=True, model=model_type)
    make_preds(df_rct_val, predictors, rct_models)
    
    obs_models = fit_models(df_obs_train, predictors, is_rct=False, model=model_type)
    make_preds(df_obs_val, predictors, obs_models)

    pr_model = fit_model(pd.concat([df_rct_train, df_obs_train]), predictors, "R")
    df_val = merge_df_val(df_rct_val, df_obs_val, predictors, pr_model, rct_models, obs_models)
    bias_res.append(df_val['b1(X)'].mean())
    for key in ['SE_Y0', 'SE_Y1', 'SE_A', 'SE_S']:
        cov_res['Pearson'][key].append(pearsonr(df_val, 'abs(b1(X))', key, df_val.shape[0]))

In [69]:
cov_res_final = defaultdict(list)
keys = ['SE_Y0', 'SE_Y1', 'SE_A', 'SE_S']
alpha = 0.01
for key in keys: 
    l = cov_res['Pearson'][key]
    # res = [x[0] for x in l if x[1] < alpha]
    # mean, standard deviation, sample size 
    mean = np.mean(l); std = np.std(l); n = len(l)
    lower = mean - 1.96 * (std / np.sqrt(n))
    upper = mean + 1.96 * (std / np.sqrt(n))
    cov_res_final[key].append(mean)
    cov_res_final[key].append(lower)
    cov_res_final[key].append(upper)
    

In [ ]:
df = pd.DataFrame.from_dict(cov_res_final, orient='index', columns=['mean', 'lower', 'upper'])
filename_save = f'./results/run_{selection_flag}_includecensored_{censored_patients_sel0}_{outcome_name}_{model_type}.csv'
print(f'Saving {filename_save}....')
df.to_csv(
    filename_save,          # File name
    sep=',',               # Delimiter (comma)
    index=True,            # Include index
    header=True,           # Include headers
    float_format='%.6f'    # Floating-point format
)
df


## Training models (1 run, $n=1$)

In [15]:
import sys
sys.path.append('../synthetic/')
from utils_data_v2 import fit_models, make_preds, merge_df_val, fit_model

In [16]:
df_ctos_train, df_ctos_val = train_test_split(df_ctos, test_size=0.25, random_state=seed)

# split into RCT and OBS
df_rct_train = df_ctos_train.query('R == 1') 
df_obs_train = df_ctos_train.query('R == 0')
df_rct_val   = df_ctos_val.query('R == 1')
df_obs_val   = df_ctos_val.query('R == 0')

In [ ]:
rct_models = fit_models(df_rct_train, predictors, is_rct=True)
make_preds(df_rct_val, predictors, rct_models)

In [ ]:
obs_models = fit_models(df_obs_train, predictors, is_rct=False)
make_preds(df_obs_val, predictors, obs_models)

In [ ]:
pr_model = fit_model(pd.concat([df_rct_train, df_obs_train]), predictors, "R")
df_val = merge_df_val(df_rct_val, df_obs_val, predictors, pr_model, rct_models, obs_models)

In [ ]:
from utils_v2 import pearsonr
from collections import defaultdict
bias_res = list() 
bias_res.append(df_val['w1(X)'].mean())
cov_res = defaultdict(lambda: defaultdict(list))
print(df_val.shape[0])
for key in ['SE_Y0', 'SE_Y1', 'SE_A', 'SE_S']:
    cov_res['Pearson'][key].append(pearsonr(df_val, 'abs(w1(X))', key, df_val.shape[0]))

In [ ]:
cov_res

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
cp = sns.color_palette("tab10")
fig, axs = plt.subplots(2, 2, figsize=(10,8)) 

for idx, key in enumerate(cov_res["Pearson"]):
    i = idx // 2
    j = idx % 2

    axs[i, j].set_title(f"Cov($w1(X)$, {key})", fontsize=16)
    axs[i, j].axhline(y=-np.log10(0.05), color='dimgray', linestyle='--', label='p = 0.05')
    
    arr = np.array(cov_res["Pearson"][key])
    hat_cov = arr[:, 0]
    log_p_val = np.clip(-np.log10(arr[:, 1]), a_min=None, a_max=5)
    axs[i, j].scatter(hat_cov, log_p_val, color=cp[idx], s=16, alpha=1)

axs[0,0].set_ylabel('-log10(p-value)', fontsize=16)
axs[1,0].set_ylabel('-log10(p-value)', fontsize=16)
axs[1,0].set_xlabel("Pearson's R", fontsize=16)
axs[1,1].set_xlabel("Pearson's R", fontsize=16)

plt.show()

In [ ]:
f = 'BAC'
prefix = 'ZD'
prefix == f[:len(prefix)]
li = ['A', 'C', 'B']
li.sort()
print(li)